# ETL — Predicción de viento para activación de seguridad en paneles solares

Este notebook cubre **únicamente el proceso ETL**: de los ficheros en crudo al dataset
horario listo para modelar. El modelado va en un notebook aparte.

## Flujo general

```
  FUENTE                          EXTRACCIÓN                    SALIDA
  ─────────────────────────────────────────────────────────────────────────────
  ~103.767 ficheros .nc     →   extraer_harmonie.py       →   harmonie_crudo.csv
  (HARMONIE-AROME, malla)       (1 fichero = 1 fila)          (103.767 filas)
                                                                     │
  Tarifa_Tabla1.dat         →   este notebook             →          │
  Tarifa_Tabla1_2022.dat        (concatena, deduplica,    →   observaciones_horarias.csv
  (anemómetro, 10 min)           agrega a horario)                    │
                                                                     │
                                                                     ▼
                                este notebook             →   dataset_horario.csv
                                (unión + tramos)              (listo para modelar)
```

## Convenciones que se aplican en todo el ETL

| Decisión | Motivo |
|---|---|
| Todo en **UTC** | Las pasadas de HARMONIE se nombran en UTC; mezclar husos desalinea las series |
| Frecuencia **horaria** | Es la resolución de HARMONIE; el anemómetro se agrega a ella |
| Dirección: **de dónde viene** el viento | Convención meteorológica estándar y la del anemómetro |
| Nunca interpolar entre huecos largos | Inventar datos a través de una caída de días falsea la validación |

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================================
RUTA = Path('.')                       # carpeta con los ficheros de entrada
HARMONIE_CRUDO = RUTA / 'harmonie_crudo.csv'
ANEMOMETRO     = [RUTA / 'Tarifa_Tabla1.dat',      # 2019-10 -> 2021-07
                  RUTA / 'Tarifa_Tabla1_2022.dat'] # 2021-06 -> 2026-08 (solapa 5 semanas)

LEAD_MIN  = 4     # h. Lead mínimo utilizable de HARMONIE: latencia de descarga y proceso
MAX_HUECO = 12    # h. Racha de huecos por encima de la cual se corta el tramo
FMT_FECHA = '%Y-%m-%d %H:%M:%S'
# ============================================================================

In [2]:
crudo = pd.read_csv(HARMONIE_CRUDO, parse_dates=['valida', 'pasada'])

print(f'{len(crudo)} filas, {crudo.valida.nunique()} horas válidas únicas')
print(f'Rango: {crudo.valida.min()} -> {crudo.valida.max()}')
print(f'Duplicados (valida, pasada): {crudo.duplicated(["valida","pasada"]).sum()}  <- debe ser 0')
print(f'\nHoras de pasada presentes (UTC):')
print(crudo.pasada.dt.hour.value_counts().sort_index().to_string())
print(f'\nLead: min {crudo.lead.min()}, mediana {crudo.lead.median():.0f}, max {crudo.lead.max()}')
print(f'\nColumnas: {list(crudo.columns)}')

209384 filas, 38185 horas válidas únicas
Rango: 2018-05-24 07:00:00 -> 2023-01-02 18:00:00
Duplicados (valida, pasada): 0  <- debe ser 0

Horas de pasada presentes (UTC):
pasada
0     52933
6     51869
12    52983
18    51599

Lead: min 1, mediana 24, max 48

Columnas: ['valida', 'pasada', 'lead', 'dominio', 'dist_grados', 'lat_malla', 'lon_malla', 'MSLP', 'SST', 'T2M', 'WSPE', 'WSPN', 'PRE', 'E', 'CLFR', 'FOG', 'SSHF', 'SLHF', 'SSRD', 'STRD', 'SH2M', 'RH2M', 'MASK', 'Z', 'SSR', 'STR']


## 1.3 Derivadas: velocidad y dirección

HARMONIE da el viento en componentes `(WSPE, WSPN)`. La velocidad es el módulo.
La dirección necesita cuidado con el signo:

```
np.arctan2(WSPE, WSPN)      -> dirección HACIA la que sopla
np.arctan2(-WSPE, -WSPN)    -> dirección de la que VIENE  ← la que usamos
```

**Verificación empírica:** comparando ambas convenciones contra el anemómetro sobre
10.954 horas coincidentes (filtrando calmas), el sesgo circular mediano es de −165,6°,
es decir ~180°. Confirma que hay que aplicar el giro.

> Nota para la memoria: AEMET indica en sus productos gráficos que representa la
> dirección *hacia donde va* el viento. El anemómetro usa la convención contraria, que
> es la estándar en observación. Conviene dejar esto escrito para no generar dudas.

In [3]:
crudo['V']   = np.hypot(crudo.WSPE, crudo.WSPN)
crudo['Dir'] = (np.degrees(np.arctan2(-crudo.WSPE, -crudo.WSPN)) + 360) % 360

print(f'Velocidad: media {crudo.V.mean():.2f}, max {crudo.V.max():.2f} m/s')
print('  (Tarifa es zona de viento fuerte por el efecto Venturi del Estrecho:')
print('   una media de 6-8 m/s es lo esperable, más alta que en un punto genérico)')
print(f'MSLP: media {crudo.MSLP.mean():.0f} Pa   T2M: media {crudo.T2M.mean():.1f} K')

Velocidad: media 6.88, max 20.46 m/s
  (Tarifa es zona de viento fuerte por el efecto Venturi del Estrecho:
   una media de 6-8 m/s es lo esperable, más alta que en un punto genérico)
MSLP: media 101772 Pa   T2M: media 291.2 K


## 1.4 Selección de la pasada operativa

Para cada hora objetivo se toma **la pasada más fresca cuyo lead sea >= `LEAD_MIN`**.
Con `LEAD_MIN = 4`, es la pasada más reciente que ya está descargada y procesada a tiempo
de que el sistema pueda actuar.

Además se guarda la **dispersión entre las pasadas disponibles** para esa hora. Es una
estimación de incertidumbre que sale gratis: cuando varias pasadas del modelo físico
discrepan mucho sobre la misma hora, la predicción es menos fiable. Es la señal natural
para escalar a decisión humana (esquema *Human-in-the-loop*).

In [4]:
apto = crudo[crudo.lead >= LEAD_MIN].sort_values(['valida', 'pasada'])

oper = apto.groupby('valida').last()                                    # pasada más fresca válida
disp = apto.groupby('valida')['V'].agg(['std', 'min', 'max', 'size'])   # dispersión entre pasadas

harm = pd.DataFrame({
    'H_vel':   oper['V'],       'H_dir':  oper['Dir'],
    'H_mslp':  oper['MSLP'],    'H_t2m':  oper['T2M'],   'H_rh2m': oper['RH2M'],
    'H_lead':  oper['lead'],
    'H_disp':  disp['std'],     'H_rango': disp['max'] - disp['min'],
    'H_nruns': disp['size'],
})
harm.index.name = 'ts'

print(f'{len(harm)} horas con pronóstico operativo (lead >= {LEAD_MIN} h)')
print(f'Lead real usado: media {harm.H_lead.mean():.1f} h, '
      f'mediana {harm.H_lead.median():.0f} h, rango [{harm.H_lead.min():.0f}, {harm.H_lead.max():.0f}]')
print(f'Pasadas disponibles por hora: media {harm.H_nruns.mean():.1f}')
print(f'Dispersión entre pasadas: media {harm.H_disp.mean():.2f} m/s')

38167 horas con pronóstico operativo (lead >= 4 h)
Lead real usado: media 8.2 h, mediana 8 h, rango [4, 48]
Pasadas disponibles por hora: media 5.1
Dispersión entre pasadas: media 0.72 m/s


---
# 2. Fuente B — Anemómetro de Tarifa (`.dat`)

## 2.1 Esquema del fichero de origen

Formato **TOA5**, propio de los dataloggers Campbell Scientific (`CR800`).
Tiene **4 líneas de cabecera**, no una:

| Línea | Contenido | ¿Se lee? |
|---|---|---|
| 0 | Metadatos del datalogger: `"TOA5","APBA03_Tarifa_Ethernet","CR800",…` | No |
| 1 | **Nombres de columna** | **Sí** |
| 2 | Unidades: `"TS","RN","m/s","Deg",…` | No |
| 3 | Tipo de agregación: `"","","WVc","WVc","Max",…` | No |
| 4+ | Datos | Sí |

Por eso se lee con `skiprows=[0, 2, 3]`: se salta la 0, 2 y 3, y se usa la 1 como cabecera.

### Columnas (13 en total)

| Columna | Unidad | Agregación | Nota |
|---|---|---|---|
| `TIMESTAMP` | — | — | Marca temporal del registro |
| `RECORD` | — | — | Contador secuencial del datalogger |
| `Vel_Viento_Avg` | m/s | WVc | **Velocidad media vectorial** ← objetivo |
| `Dir_Viento_Avg` | Deg | WVc | Dirección media vectorial (de dónde viene) |
| `Dir_Viento_SD` | Deg | WVc | Desviación típica de la dirección |
| `Vel_Viento_3s_Max` | m/s | Max | **Racha máxima de 3 s** |
| `Vel_Viento_3s_TMx` | m/s | TMx | Instante de la racha máxima |
| `Dir_Viento_3s` | Grados | SMM | Dirección en la racha máxima |
| `Vel_Viento_SD`, `Vel_Viento_Std` | m/s | Std | Desviación típica de la velocidad |
| `Visibility_Avg` | — | Avg | Visibilidad |
| `SWS200_Status`, `SWS200_WMO` | — | Smp | Estado del sensor de visibilidad |

### Los datos vienen en DOS ficheros que solapan

| Fichero | Registros | Rango |
|---|---|---|
| `Tarifa_Tabla1.dat` | 91.512 | 2019-10-09 → 2021-07-06 |
| `Tarifa_Tabla1_2022.dat` | 260.591 | 2021-06-01 → 2026-08-17 |

Solapan entre 2021-06-01 y 2021-07-06 (**5.103 registros comunes**). Se ha verificado que
en ese tramo **los valores son idénticos al 100%**, así que concatenar y deduplicar por
`TIMESTAMP` es seguro y no hay que decidir qué fichero tiene prioridad.

Usar solo el primer fichero dejaba fuera julio-octubre de 2021: unas **1.800 horas
adicionales (+14%)**, y precisamente al final de la serie, que es el periodo que acabará
siendo el conjunto de test.

> El periodo útil lo limita la intersección con HARMONIE (que acaba en 2021-10-01), no el
> anemómetro. Los datos posteriores a esa fecha quedan disponibles para una validación
> futura si se reextrae HARMONIE con más cobertura.

### Características de los datos

* Muestreo cada **10 minutos** (no horario).
* **Los nulos vienen como el texto `'NAN'`**, no como celdas vacías. Si no se convierten
  explícitamente con `pd.to_numeric(..., errors='coerce')`, pandas lee toda la columna
  como texto y las comparaciones numéricas fallan en silencio.
* 15.557 registros con `NAN` en las variables de viento (~17%).

> **`Vel_Viento_3s_Max` está disponible y no se usa como objetivo.** Merece una decisión
> explícita en la memoria: si el umbral operativo real de activación se aplica a la racha
> y no a la media horaria, el objetivo debería ser esta columna — y la clase positiva
> sería bastante más minoritaria.

In [5]:
COLS = ['TIMESTAMP', 'Vel_Viento_Avg', 'Dir_Viento_Avg', 'Vel_Viento_3s_Max']

def leer_toa5(ruta):
    """Lee un fichero TOA5: 4 líneas de cabecera, nulos como texto 'NAN'."""
    d = pd.read_csv(ruta, skiprows=[0, 2, 3], low_memory=False)[COLS].copy()
    d['TIMESTAMP'] = pd.to_datetime(d['TIMESTAMP'])
    # CRÍTICO: los nulos son el texto 'NAN'. Sin errors='coerce' la columna queda como str
    # y las comparaciones numéricas fallan en silencio.
    for c in COLS[1:]:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

partes = []
for ruta in ANEMOMETRO:
    d = leer_toa5(ruta)
    print(f'{ruta.name}: {len(d)} registros, {d.TIMESTAMP.min()} -> {d.TIMESTAMP.max()}')
    partes.append(d)

anem = pd.concat(partes, ignore_index=True).sort_values('TIMESTAMP')

# Verificación del solape ANTES de deduplicar: si los ficheros discreparan,
# habría que decidir cuál tiene prioridad en vez de quedarse con uno sin mirar.
dups = anem[anem.duplicated('TIMESTAMP', keep=False)]
if len(dups):
    disc = dups.groupby('TIMESTAMP')['Vel_Viento_Avg'].nunique()
    print(f'\nSolape: {dups.TIMESTAMP.nunique()} instantes en más de un fichero')
    print(f'  instantes con valores DISCREPANTES: {(disc > 1).sum()}  <- debe ser 0')

anem = anem.drop_duplicates('TIMESTAMP', keep='last').reset_index(drop=True)

print(f'\n{len(anem)} registros tras concatenar y deduplicar')
print(f'Rango: {anem.TIMESTAMP.min()} -> {anem.TIMESTAMP.max()}')
print(f'Paso de muestreo más frecuente: {anem.TIMESTAMP.diff().mode()[0]}')
print(f'\nNulos por columna:')
print(anem[COLS[1:]].isna().sum().to_string())

Tarifa_Tabla1.dat: 91512 registros, 2019-10-09 14:10:00 -> 2021-07-06 10:20:00
Tarifa_Tabla1_2022.dat: 260591 registros, 2021-06-01 00:00:00 -> 2026-08-17 08:00:00

Solape: 5106 instantes en más de un fichero
  instantes con valores DISCREPANTES: 0  <- debe ser 0

346997 registros tras concatenar y deduplicar
Rango: 2019-10-09 14:10:00 -> 2026-08-17 08:00:00
Paso de muestreo más frecuente: 0 days 00:10:00

Nulos por columna:
Vel_Viento_Avg       15636
Dir_Viento_Avg       15636
Vel_Viento_3s_Max    15636


## 2.2 Control de calidad y agregación a horario

Dos filtros, ambos con justificación explícita:

* `Vel_Viento_Avg >= 0` — se usa **>= 0, no > 0**: las calmas son observaciones válidas,
  no errores. Filtrar `> 0` eliminaría silenciosamente los periodos sin viento.
* `0 <= Dir_Viento_Avg <= 360` — descarta valores fuera del rango físico del sensor.

### Agregación a horario: la dirección necesita media circular

La dirección del viento es una **magnitud circular**: 0° y 360° son el mismo punto. La media
aritmética da resultados sin sentido en ese entorno — el promedio de 350° y 10° da **180°**,
exactamente el rumbo opuesto al correcto (0°).

La forma correcta es descomponer cada observación en componentes vectoriales, promediar las
componentes, y recomponer el ángulo:

```
u = −velocidad · sin(dirección)      (componente este)
v = −velocidad · cos(dirección)      (componente norte)
                                      El signo negativo mantiene la convención
                                      "de dónde viene" el viento.

dirección_horaria = atan2(−ū, −v̄)
```

Al ponderar por la velocidad (usando `u`, `v` en vez de vectores unitarios) se obtiene la
**dirección resultante**, que es lo que recomienda la OMM: las rachas fuertes pesan más que
las calmas, que es el comportamiento deseable para un sistema de seguridad.

### Velocidad: media escalar, no vectorial

`obs_vel` se mantiene como **media escalar** de la velocidad, no como el módulo del vector
promedio. Son cosas distintas: si el viento gira 180° a mitad de hora, la media vectorial se
acerca a cero mientras que la escalar refleja que ha soplado fuerte todo el rato. Para un
umbral de seguridad interesa la magnitud del viento, no el transporte neto.

De la relación entre ambas sale gratis un indicador útil:

```
constancia = |vector promedio| / media escalar        ∈ [0, 1]
```

Vale 1 con viento perfectamente estable en dirección y baja hacia 0 cuando es racheado y
cambiante. Se guarda como `obs_constancia`.

In [6]:
anem = anem.dropna(subset=['Vel_Viento_Avg', 'Dir_Viento_Avg'])
anem = anem[(anem.Vel_Viento_Avg >= 0) & anem.Dir_Viento_Avg.between(0, 360)]
print(f'{len(anem)} registros tras el control de calidad')

# Componentes vectoriales (signo negativo = convención "de dónde viene")
rad = np.radians(anem.Dir_Viento_Avg)
anem['u'] = -anem.Vel_Viento_Avg * np.sin(rad)
anem['v'] = -anem.Vel_Viento_Avg * np.cos(rad)

obs = (anem.set_index('TIMESTAMP').sort_index()
           .resample('h')
           .agg({'Vel_Viento_Avg': 'mean',      # media ESCALAR de la velocidad
                 'Vel_Viento_3s_Max': 'max',
                 'u': 'mean', 'v': 'mean'}))    # media VECTORIAL para la dirección
obs.columns = ['obs_vel', 'obs_racha', 'u', 'v']
obs.index.name = 'ts'

# Dirección resultante a partir de las componentes promediadas
obs['obs_dir'] = (np.degrees(np.arctan2(-obs.u, -obs.v)) + 360) % 360
# Constancia: |vector promedio| / media escalar. 1 = dirección estable, 0 = muy cambiante
obs['obs_constancia'] = (np.hypot(obs.u, obs.v) / obs.obs_vel).where(obs.obs_vel > 0.1)
obs = obs.drop(columns=['u', 'v'])[['obs_vel', 'obs_dir', 'obs_racha', 'obs_constancia']]

print(f'{len(obs)} horas, {obs.obs_vel.isna().sum()} sin observación')
print(f'Velocidad horaria: media {obs.obs_vel.mean():.2f}, max {obs.obs_vel.max():.2f} m/s')
print(f'Constancia direccional: media {obs.obs_constancia.mean():.3f} '
      f'(1 = dirección estable dentro de la hora)')

# Comprobación de que la media circular no se rompe cerca de 0°/360°
cerca_norte = obs.obs_dir[(obs.obs_dir > 350) | (obs.obs_dir < 10)]
print(f'\nHoras con dirección cerca del norte (>350° o <10°): {len(cerca_norte)}')
print('  (con media aritmética estas horas aparecerían erróneamente en torno a 180°)')

obs.to_csv(RUTA / 'observaciones_horarias.csv', date_format=FMT_FECHA)
print(f'\nGuardado: observaciones_horarias.csv')

331361 registros tras el control de calidad
60091 horas, 4694 sin observación
Velocidad horaria: media 8.91, max 28.33 m/s
Constancia direccional: media 0.986 (1 = dirección estable dentro de la hora)

Horas con dirección cerca del norte (>350° o <10°): 603
  (con media aritmética estas horas aparecerían erróneamente en torno a 180°)

Guardado: observaciones_horarias.csv


---
# 3. Unión y construcción de tramos continuos

## 3.1 Unión

`inner join` por hora: solo se conservan las horas con **pronóstico y observación**.
El periodo útil queda limitado por el anemómetro (arranca en octubre de 2019, mientras
que HARMONIE cubre desde mayo de 2018).

## 3.2 Por qué hacen falta "tramos"

La serie tiene huecos: fallos del sensor, cortes de comunicación, pasadas no descargadas.
Tratar la serie como continua rompería dos cosas:

* Un `shift(1)` a través de un hueco de tres días tomaría como "hora anterior" un valor
  de tres días atrás.
* Interpolar a través de un hueco largo **inventa** datos que luego se usarían para
  validar el modelo.

La solución es partir la serie en **tramos continuos**: cada racha de huecos de más de
`MAX_HUECO` horas corta el tramo. Los `shift` y las ventanas se calculan siempre
**dentro de cada tramo**, nunca a través de la frontera.

## 3.3 Interpolación: `limit_area='inside'`

Los huecos cortos (≤ `MAX_HUECO`) sí se interpolan, pero con `limit_area='inside'`,
que **solo rellena entre valores conocidos** y nunca extrapola hacia los bordes del tramo.

> La versión anterior de este ETL usaba `limit_direction='both'`, que sí extrapola hacia
> atrás. Eso puede arrastrar información posterior a través de la frontera del split
> train/test — una fuga temporal sutil pero real.

In [7]:
df = obs.join(harm, how='inner').asfreq('h')
print(f'{len(df)} horas tras la unión')
print(f'Rango: {df.index.min()} -> {df.index.max()}')

# Una hora es válida si tiene AMBAS fuentes
valido = df.obs_vel.notna() & df.H_vel.notna()

# Tamaño de cada racha de NaN: se agrupa por el contador acumulado de VÁLIDOS,
# de modo que todos los NaN consecutivos entre dos válidos caen en el mismo grupo.
hueco = (~valido).groupby(valido.cumsum()).transform('sum').where(~valido, 0)
df['tramo'] = (hueco > MAX_HUECO).cumsum()

# Se descartan los tramos que no contienen ninguna observación válida
df = df[valido.groupby(df.tramo).transform('sum') > 0].copy()

print(f'{df.tramo.nunique()} tramos continuos')
print('\nDuración de cada tramo (horas):')
print(df.groupby('tramo').size().describe().round(0).to_string())

28349 horas tras la unión
Rango: 2019-10-09 14:00:00 -> 2023-01-02 18:00:00
13 tramos continuos

Duración de cada tramo (horas):
count      13.0
mean     1825.0
std      2610.0
min         3.0
25%        16.0
50%       297.0
75%      2431.0
max      8800.0


In [8]:
# Interpolación SOLO de huecos interiores de cada tramo (nunca extrapola hacia los bordes)
for c in [c for c in df.columns if c != 'tramo']:
    df[c] = df.groupby('tramo')[c].transform(lambda s: s.interpolate(limit_area='inside'))

df = df.dropna(subset=['obs_vel', 'H_vel'])
df['H_disp'] = df['H_disp'].fillna(0.0)   # 1 sola pasada disponible -> dispersión nula

print(f'{len(df)} horas finales, {df.tramo.nunique()} tramos')
print(f'Rango: {df.index.min()} -> {df.index.max()}')
print(f'\nNulos restantes:')
print(df.isna().sum()[lambda s: s > 0].to_string() or '  ninguno')

23719 horas finales, 13 tramos
Rango: 2019-10-09 14:00:00 -> 2023-01-02 18:00:00

Nulos restantes:
Series([], )


---
# 4. Salida — `dataset_horario.csv`

## Esquema

| Columna | Unidad | Fuente | Descripción |
|---|---|---|---|
| `ts` (índice) | datetime UTC | — | Hora objetivo |
| `obs_vel` | m/s | anemómetro | **Velocidad media horaria observada** (objetivo) |
| `obs_dir` | grados | anemómetro | Dirección resultante (de dónde viene), **media circular** |
| `obs_racha` | m/s | anemómetro | Racha máxima de 3 s en la hora |
| `obs_constancia` | 0-1 | anemómetro | Estabilidad direccional dentro de la hora |
| `H_vel` | m/s | HARMONIE | Velocidad pronosticada |
| `H_dir` | grados | HARMONIE | Dirección pronosticada (de dónde viene) |
| `H_mslp` | Pa | HARMONIE | Presión a nivel del mar |
| `H_t2m` | K | HARMONIE | Temperatura a 2 m |
| `H_rh2m` | 0-1 | HARMONIE | Humedad relativa a 2 m |
| `H_lead` | h | HARMONIE | **Lead real** de la pasada usada (>= `LEAD_MIN`) |
| `H_disp` | m/s | HARMONIE | Desv. típica entre pasadas disponibles (incertidumbre) |
| `H_rango` | m/s | HARMONIE | Máx − mín entre pasadas |
| `H_nruns` | — | HARMONIE | Nº de pasadas disponibles para esa hora |
| `tramo` | — | derivado | Identificador de tramo continuo |

## Lo que este ETL deliberadamente **no** hace

Todo esto pertenece al notebook de modelado, no aquí:

* **No crea lags ni ventanas.** Dependen del horizonte de predicción elegido, que es una
  decisión de modelado.
* **No parte en train/val/test.** El split va donde se entrena.
* **No define la variable de clasificación.** El umbral de activación (11,5 m/s) y la
  binarización se aplican al modelar, para poder probar varios umbrales sin rehacer el ETL.
* **No normaliza ni escala.** Los escaladores deben ajustarse solo con el conjunto de
  entrenamiento; hacerlo aquí sería una fuga.

In [9]:
SALIDA = RUTA / 'dataset_horario.csv'
df.to_csv(SALIDA, date_format=FMT_FECHA)

print(f'Guardado: {SALIDA}')
print(f'  {len(df)} filas x {len(df.columns)} columnas')
print(f'  {df.index.min()} -> {df.index.max()}')
print(f'  {df.tramo.nunique()} tramos continuos')
display(df.head())

Guardado: dataset_horario.csv
  23719 filas x 14 columnas
  2019-10-09 14:00:00 -> 2023-01-02 18:00:00
  13 tramos continuos


,obs_vel,obs_dir,obs_racha,obs_constancia,H_vel,H_dir,H_mslp,H_t2m,H_rh2m,H_lead,H_disp,H_rango,H_nruns,tramo
ts,,,,,,,,,,,,,,
2019-10-09 14:00:00,12.936853,83.517586,16.15333,0.999833,8.274787,94.435879,101528.0,298.03,-120.422226,14.0,0.145022,0.280831,3.0,0
2019-10-09 15:00:00,11.972688,82.715607,15.19333,0.999327,8.417244,96.685977,101470.0,297.21,-116.260834,15.0,0.172515,0.343057,3.0,0
2019-10-09 16:00:00,12.602606,90.119298,14.86667,0.996749,8.299596,93.938062,101474.0,296.21,-103.051109,4.0,0.177993,0.428459,4.0,0
2019-10-09 17:00:00,13.767855,92.977809,17.84667,0.998167,8.116711,92.330109,101454.0,295.28,-91.937225,5.0,0.159675,0.356395,4.0,0
2019-10-09 18:00:00,10.762686,80.748099,12.67667,0.997792,6.972589,88.438522,101484.0,294.15,-79.245003,6.0,0.317148,0.719525,4.0,0


## Verificación final

Comprobaciones que deben pasar antes de dar el dataset por bueno.

In [10]:
fallos = []

if df.index.duplicated().any():
    fallos.append('Hay horas duplicadas en el índice')
if not df.index.is_monotonic_increasing:
    fallos.append('El índice no está ordenado')
if (df.H_lead < LEAD_MIN).any():
    fallos.append(f'Alguna fila tiene lead < {LEAD_MIN} h')
if df[['obs_vel', 'H_vel']].isna().any().any():
    fallos.append('Quedan nulos en las columnas principales')
if not df.obs_vel.between(0, 60).all():
    fallos.append('obs_vel fuera del rango físico [0, 60] m/s')
if not df.H_vel.between(0, 60).all():
    fallos.append('H_vel fuera del rango físico [0, 60] m/s')
if not df.obs_dir.between(0, 360).all():
    fallos.append('obs_dir fuera de [0, 360]')

# Ninguna ventana de 6 h debe cruzar la frontera de un tramo
saltos = df.groupby('tramo').apply(
    lambda g: (g.index.to_series().diff().dropna() != pd.Timedelta('1h')).sum(),
    include_groups=False)
if saltos.sum() > 0:
    fallos.append(f'{saltos.sum()} discontinuidades horarias DENTRO de algún tramo')

if fallos:
    print('FALLOS:')
    for f in fallos:
        print(f'  - {f}')
else:
    print('Todas las comprobaciones pasan.')

print(f'\nResumen del dataset:')
print(f'  horas: {len(df)}')
print(f'  cobertura temporal: {(df.index.max() - df.index.min()).days} días')
print(f'  densidad: {100*len(df)/((df.index.max()-df.index.min()).total_seconds()/3600+1):.1f}% '
      f'de las horas del periodo')
print(f'  lead medio de HARMONIE: {df.H_lead.mean():.1f} h')

Todas las comprobaciones pasan.

Resumen del dataset:
  horas: 23719
  cobertura temporal: 1181 días
  densidad: 83.7% de las horas del periodo
  lead medio de HARMONIE: 7.4 h
